In [3]:
!pip install mlflow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 119.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 764.2/764.2 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 24.5 MB/s eta 0:00:00


In [10]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fraud-detection")


2025/12/15 10:45:12 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/mlruns/1', creation_time=1765795512381, experiment_id='1', last_update_time=1765795512381, lifecycle_stage='active', name='fraud-detection', tags={}>

In [11]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score
import os
import joblib

In [12]:
import sys
print(sys.executable)

/usr/bin/python3


In [13]:
import sys, platform
print("Python executable:", sys.executable)
print("Platform:", platform.system())


Python executable: /usr/bin/python3
Platform: Linux


In [14]:
import mlflow
print(mlflow.__version__)


3.7.0


In [15]:
import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib
import os

# Dummy data (replace with real later)
X = np.random.rand(1000, 10)
y = np.random.randint(0, 2, 1000)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

with mlflow.start_run():
    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, preds)

    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 12)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("roc_auc", auc)

    mlflow.sklearn.log_model(model, "fraud_model")

    os.makedirs("../backend/models", exist_ok=True)
    joblib.dump(model, "../backend/models/fraud_model.pkl")

print("✅ Training complete. Model saved.")


2025/12/15 10:45:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ Training complete. Model saved.


In [16]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name("Default")

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
    max_results=1
)

run_id = runs[0].info.run_id
model_uri = f"runs:/{run_id}/fraud_model"

client.create_model_version(
    name="FraudDetectionRF",
    source=model_uri,
    run_id=run_id
)

print("✅ Model registered from latest run")



✅ Model registered from latest run


In [17]:
import mlflow
import mlflow.sklearn

MODEL_NAME = "FraudDetectionRF"

def load_latest_model():
    model_uri = f"models:/{MODEL_NAME}/latest"
    model = mlflow.sklearn.load_model(model_uri)
    return model
